# Parameter Information
1. So    :   initial value, that will be the last value of the historical data
2. dt    :   time increment, 1 month in our case. It is determined by the frequency of the historical data.For example, if the historical data is daily, dt should be 1/30 (1 month = 30 days). But it should be consistent with T.
3. T     :   length of the prediction time horizon, how many time points to predict. It should be consistent with dt. For example, if dt is 1 month and we want to predict for 6 months, T should be 6.
4. N     :   number of time points in the prediction time horizon, it is calculated by T/dt. For example, if T is 6 months and dt is 1 month, N should be 6. It is the number of time points within our prediction time horizon which should be consistent with our historical data in terms of time increment magnitude.
5. t     :   array for time points in the prediction time horizon [1, 2, 3, .. , N]. This is an array where we show the time progression in our model. It is like a time ticker where we measure time by counting the number of time points elapsed.
6. mu    :  Mean change in daily crime counts over the selected historical period. It is the average change in the historical data. It is calculated by taking the difference between each consecutive data point, summing them up and dividing by the number of data points. We will then use mu in our drift component calculation. It will have an effect on the long-term movement of the daliy number of crimes
7. sigma :   standard deviation of historical data (number of crimes per day). sigma will contribute by scaling the magnitude of random shock so that the small fluctuations occur in accordance with the historical volatility of the data.
8. b     :   array for brownian increments. Here array b, for each corresponding prediction time point, stores a random number coming from the standard normal distribution. These random numbers will add the random shocks. b is the random shock being applied to the data at a time point when predicting the data of the NEXT time point. So, suppose, at time point 3, the daily number of crimes is S_3. When predicting time point 4, b(4) is applied to S_3 as the random shock.
9. W     :   array for brownian path and it determines how the daily number of crimes fluctuate from beginning time point (So) to some other time point t. It means that it includes the effects of all the random shocks since the beginning of the prediction time horizon. It is the total effect of randomness incorporated into So(initial stock price) until the specific time point we are concerned with. So, suppose, we are predicting time point 4, we need to apply all the random shocks up-to and including time point 4 to So. Therefore, instead of b(4), we use W(4) which is the cumulative sum of array b elements with index less than or equal to 4.


## Import Data and Libraries

In [2]:
import folium
from folium.plugins import MarkerCluster
import calendar
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import learning_curve
import numpy as np
from sklearn.model_selection import train_test_split, KFold,GridSearchCV, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import mean_squared_error
import os
import pandas as pd
import plotly.express as px

In [7]:
# Fájl elérési útja
csv_path = 'data/Number_of_crimes_per_day.csv'

# Beolvasás
df = pd.read_csv(csv_path)

In [8]:
# Create a new dataframe which contains data only for first district
df1 = df[df['District'] == 1].copy()
df1 = df1.drop(columns=['count', 'Number_of_days_in_month', 'District'])

In [16]:
# The historical data will be the data from 2001 to 2024
historical_data = df1[df1['Year'].between(2001, 2024)]
# We will predict to 2025
prediction_data = df1[df1['Year'] == 2025]
prediction_data.reset_index(drop=True, inplace=True)
validation_data = prediction_data['Number of Crimes Per Day'].copy()
prediction_data = prediction_data.drop(columns=['Number of Crimes Per Day'])


## Geometric Brownian Motion Model Implementation - Define Parameters

In [17]:
# Initial value, that will be the last value of the historical data
So = historical_data['Number of Crimes Per Day'].values[-1]
So


np.float64(35.774193548387096)

In [19]:
# Time increment, 1 month in our case
dt = 1 # 1 month
# Length of the prediction time horizon, how many time points to predict
T = 9  # 9 months
# Number of time points in the prediction time horizon
N = int(T / dt)  # 9 time points
# Array for time points in the prediction time horizon [1, 2, 3, .. , N]
t = np.arange(1, int(N) + 1)
print(t)

[1 2 3 4 5 6 7 8 9]


In [23]:
# Calculate mu- first we calculate the rate of relative daily changes in the historical data ((actual data - previous data)/previous data)
Rate_of_dailychange = np.diff(historical_data['Number of Crimes Per Day'].values) / historical_data['Number of Crimes Per Day'].values[:-1]
print(Rate_of_dailychange)
#Calculate mu
mu = np.mean(Rate_of_dailychange)
mu


[-4.29782082e-02  7.30837790e-02 -6.35658915e-02 -5.78936125e-02
  5.44217687e-02  9.91935484e-02 -8.80410858e-03 -1.17345176e-01
  6.41806899e-02  3.65904912e-02 -1.50089958e-01  3.59570662e-01
  2.26503759e-02 -1.19933830e-01  1.21710526e-01 -1.46627566e-02
  4.90520282e-02  1.18734895e-02  2.49221184e-03  1.86451212e-03
 -9.30521092e-03 -1.09016907e-01 -1.46883126e-02  3.20970043e-02
  1.83877974e-02  3.28397276e-02 -2.70915462e-02 -2.01021993e-02
  1.01699977e-01 -1.34912526e-02 -2.79010780e-02 -3.07023266e-02
  4.31154381e-02 -9.13333333e-02 -2.08979244e-02 -3.69833212e-02
 -5.17760698e-02  1.77693192e-01 -8.02427512e-02  2.19941349e-02
  1.19320899e-01  6.19525742e-02  7.24200362e-03 -9.66846415e-02
  3.40710606e-02  3.39961514e-02 -9.30521092e-02  9.91792066e-02
 -1.44323940e-01  2.68557478e-02  3.18696884e-02  6.45161290e-02
  6.13152805e-02 -1.40331693e-02 -3.69685767e-02  6.22734058e-03
 -2.84430503e-02 -1.06653578e-01 -2.71299846e-02 -3.68975904e-02
 -1.66424662e-02  6.78100

np.float64(0.005560059471798539)